# Vehicle Detection and Counting using YOLOv8

This project implements vehicle detection and counting using YOLOv8 CNN algorithm. The system can detect and count four types of vehicles: Motor, Mobil, and Person.

## Project Overview
- Algorithm: YOLOv8 (You Only Look Once)
- Classes: Motor, Mobil, Person
- Dataset: Custom vehicle dataset from Roboflow
- Training: 100 epochs with early stopping
- Output: Detection count and accuracy metrics

## 1. Environment Setup and Library Installation

Install required libraries including Ultralytics YOLOv8, Roboflow for dataset management, and visualization tools.

## 0. Google Colab Setup (Optional)

Run this cell only if you are using Google Colab to mount Google Drive for saving outputs.

In [ ]:
import sys
import os

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

    drive_dir = '/content/drive/MyDrive/UAS_COMVIS'
    os.makedirs(drive_dir, exist_ok=True)
    os.chdir(drive_dir)

    print("Google Drive mounted successfully")
    print(f"Working directory: {os.getcwd()}")
    print("All outputs will be saved to Google Drive")
else:
    print("Running on Local Environment (Mac)")
    print(f"Working directory: {os.getcwd()}")
    print("\nNOTE: If you have trained models in Google Drive, download them to:")
    print(f"  {os.getcwd()}/output/models/")
    print("\nOr mount Google Drive using Google Drive Desktop app")

In [ ]:
import sys
import os
import shutil
from pathlib import Path

print(f"Python version: {sys.version}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
import sys

if 'google.colab' in sys.modules:
    print("Running on Google Colab")
    get_ipython().system('pip install ultralytics roboflow opencv-python matplotlib seaborn pillow pyyaml')
else:
    print("Running on Local Environment")
    get_ipython().run_line_magic('pip', 'install ultralytics roboflow opencv-python matplotlib seaborn pillow pyyaml')

## 2. Import Required Libraries

Import all necessary libraries for data processing, model training, and visualization.

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
from roboflow import Roboflow
from PIL import Image
import json
from datetime import datetime
import pandas as pd
from pathlib import Path
import glob

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU for training")

## 3. Download Dataset from Roboflow

Download vehicle dataset from Roboflow. The dataset contains images of Motor, Mobil, Bus, and Truck with annotations in YOLO format.

Note: You need to get your API key from Roboflow website and replace the workspace/project details below.

In [ ]:
import os
from pathlib import Path

# Check if dataset already exists locally
local_dataset_path = Path(os.getcwd()) / 'mobil-motor-orang-2-9'

if local_dataset_path.exists():
    print(f"Using existing local dataset: {local_dataset_path}")

    class DatasetLocation:
        location = str(local_dataset_path)

    dataset = DatasetLocation()

    # Verify dataset structure
    train_imgs = len(list((local_dataset_path / 'train' / 'images').glob('*.*'))) if (local_dataset_path / 'train' / 'images').exists() else 0
    train_lbls = len(list((local_dataset_path / 'train' / 'labels').glob('*.txt'))) if (local_dataset_path / 'train' / 'labels').exists() else 0

    print(f"Train images: {train_imgs} files")
    print(f"Train labels: {train_lbls} files")
else:
    print("Downloading dataset from Roboflow...")
    from roboflow import Roboflow
    rf = Roboflow(api_key="0uJbUlaGa8dYYZ1nR1bx")
    project = rf.workspace("yolo-0fyiq").project("mobil-motor-orang-2")
    version = project.version(9)
    dataset = version.download("yolov8")

print(f"\nDataset location: {dataset.location}")

## 4. Data Exploration and Visualization

Explore the dataset structure and visualize sample images with annotations.

In [ ]:
dataset_path = Path(dataset.location)
data_yaml_path = dataset_path / "data.yaml"

print(f"Dataset structure:")
for split in ['train', 'valid', 'test']:
    split_path = dataset_path / split / 'images'
    if split_path.exists():
        num_images = len(list(split_path.glob('*.jpg'))) + len(list(split_path.glob('*.png')))
        print(f"{split.capitalize()}: {num_images} images")
    else:
        print(f"{split.capitalize()}: Not found")

print(f"\nData YAML path: {data_yaml_path}")

In [ ]:
import yaml

with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("Dataset Configuration:")
print(f"Number of classes: {data_config['nc']}")
print(f"Class names: {data_config['names']}")
print(f"Train path: {data_config.get('train', 'N/A')}")
print(f"Validation path: {data_config.get('val', 'N/A')}")
print(f"Test path: {data_config.get('test', 'N/A')}")

## 4.1. Fix Dataset Configuration

Check and fix data.yaml if class names are missing or incorrectly formatted.

In [ ]:
# Validate class configuration instead of silently overwriting it.
if data_config.get('nc', 0) == 0 or not data_config.get('names'):
    raise ValueError(
        "data.yaml tidak punya class names. Perbaiki export Roboflow dulu; "
        "jangan hardcode 2 kelas karena label file masih bisa memakai 3 kelas."
    )

names = data_config['names']
if isinstance(names, dict):
    names = [names[i] for i in sorted(names)]
    data_config['names'] = names

class_ids = set()
for split in ['train', 'valid', 'test']:
    label_dir = dataset_path / split / 'labels'
    if not label_dir.exists():
        continue
    for lbl in label_dir.glob('*.txt'):
        for line in lbl.read_text(errors='ignore').splitlines():
            if line.strip():
                class_ids.add(int(line.split()[0]))

max_id = max(class_ids) if class_ids else -1
if max_id >= data_config['nc']:
    raise ValueError(
        f"Label file berisi class id {max_id}, tapi data.yaml hanya punya "
        f"{data_config['nc']} kelas ({data_config['names']})."
    )

print(f"Dataset OK: {data_config['nc']} classes -> {data_config['names']}")
print(f"Class ids found in labels: {sorted(class_ids)}")


## 4.2. Create Validation and Test Splits

Split the training data into train, validation, and test sets if they don't exist.

In [ ]:
import random
import shutil
from pathlib import Path

train_images_dir = dataset_path / 'train' / 'images'
train_labels_dir = dataset_path / 'train' / 'labels'
valid_dir = dataset_path / 'valid'
test_dir = dataset_path / 'test'

def split_exists(split_dir):
    img_dir = split_dir / 'images'
    return img_dir.exists() and any(img_dir.glob('*.*'))

have_valid = split_exists(valid_dir)
have_test = split_exists(test_dir)

# Pakai split resmi Roboflow. Hanya buat split yang benar-benar belum ada.
if have_valid and have_test:
    print("Using the existing Roboflow train / valid / test split (no re-splitting).")
else:
    print("One split is missing -> creating only the missing split from train.")
    print(f"  valid exists: {have_valid} | test exists: {have_test}")

    all_images = sorted(train_images_dir.glob('*.*'))
    random.Random(42).shuffle(all_images)

    total = len(all_images)
    n_valid = 0 if have_valid else max(1, int(0.15 * total))
    n_test = 0 if have_test else max(1, int(0.10 * total))
    valid_set = all_images[:n_valid]
    test_set = all_images[n_valid:n_valid + n_test]

    for split_name, split_data in [('valid', valid_set), ('test', test_set)]:
        if not split_data:
            continue
        split_img_dir = dataset_path / split_name / 'images'
        split_lbl_dir = dataset_path / split_name / 'labels'
        split_img_dir.mkdir(parents=True, exist_ok=True)
        split_lbl_dir.mkdir(parents=True, exist_ok=True)

        for img_path in split_data:
            shutil.move(str(img_path), str(split_img_dir / img_path.name))
            label_path = train_labels_dir / f"{img_path.stem}.txt"
            if label_path.exists():
                shutil.move(str(label_path), str(split_lbl_dir / label_path.name))

    print(f"Created -> valid: {len(valid_set)} | test: {len(test_set)} | "
          f"remaining train: {len(all_images) - len(valid_set) - len(test_set)}")

data_config['train'] = str(dataset_path / 'train' / 'images')
data_config['val'] = str(dataset_path / 'valid' / 'images')
data_config['test'] = str(dataset_path / 'test' / 'images')

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

for split in ['train', 'valid', 'test']:
    img_dir = dataset_path / split / 'images'
    lbl_dir = dataset_path / split / 'labels'
    n_img = len(list(img_dir.glob('*.*'))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f"{split.capitalize()}: {n_img} images, {n_lbl} labels")

print("Updated data.yaml with absolute paths")


In [ ]:
train_images = list((dataset_path / 'train' / 'images').glob('*.*'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, img_path in enumerate(train_images):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    label_path = img_path.parent.parent / 'labels' / f"{img_path.stem}.txt"

    if label_path.exists():
        with open(label_path, 'r') as f:
            annotations = f.readlines()

        h, w = img.shape[:2]
        for ann in annotations:
            parts = ann.strip().split()
            if len(parts) >= 5:
                class_id = int(parts[0])
                x_center, y_center, width, height = map(float, parts[1:5])

                x1 = int((x_center - width/2) * w)
                y1 = int((y_center - height/2) * h)
                x2 = int((x_center + width/2) * w)
                y2 = int((y_center + height/2) * h)

                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(img, data_config['names'][class_id], (x1, y1-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    axes[idx].imshow(img)
    axes[idx].axis('off')
    axes[idx].set_title(f"Sample {idx+1}")

plt.tight_layout()

# Save path based on environment
if 'google.colab' in sys.modules:
    save_path = '/content/drive/MyDrive/UAS_COMVIS/output/sample_images.png'
else:
    save_path = 'output/results/sample_images.png'
    os.makedirs('output/results', exist_ok=True)

plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved to: {save_path}")
plt.show()

## 5. Model Training with YOLOv8

Train YOLOv8 model with custom vehicle dataset. Using pre-trained weights as starting point with transfer learning approach. Training configuration includes early stopping with patience of 3 epochs.

In [ ]:
model = YOLO('yolov8n.pt')

print("Model loaded successfully")
print(f"Model type: YOLOv8n")
print(f"Pre-trained on COCO dataset")

In [ ]:
results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    patience=20,
    batch=16,
    imgsz=960,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    project='output/models',
    name='vehicle_detection',
    exist_ok=True,
    pretrained=True,
    optimizer='Adam',
    lr0=0.001,
    save=True,
    plots=True,
    verbose=True
)

print("Training completed")
print(f"Best model saved at: output/models/vehicle_detection/weights/best.pt")

In [ ]:
import shutil, os
from pathlib import Path
from ultralytics import YOLO

folder = Path('/content/drive/MyDrive/UAS_COMVIS/runs/detect/output/models/vehicle_detection/weights')

print("Isi folder:", os.listdir(folder))

shutil.copy(folder / 'last.pt', folder / 'best.pt')

print("Ukuran best.pt sekarang:", (folder / 'best.pt').stat().st_size, "bytes")

best_model = YOLO(str(folder / 'best.pt'))
print("MODEL BERHASIL DIMUAT")

## 6. Training Results Visualization

Visualize training metrics including loss curves and performance metrics across epochs.

In [ ]:
if 'google.colab' in sys.modules:
    training_results_path = Path('/content/drive/MyDrive/UAS_COMVIS/output/models/vehicle_detection')
else:
    training_results_path = Path('output/models/vehicle_detection')

result_images = ['results.png', 'confusion_matrix.png', 'F1_curve.png', 'PR_curve.png']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, img_name in enumerate(result_images):
    img_path = training_results_path / img_name
    if img_path.exists():
        img = plt.imread(str(img_path))
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(img_name.replace('_', ' ').replace('.png', '').title())
    else:
        axes[idx].text(0.5, 0.5, f'{img_name} not found',
                       ha='center', va='center', fontsize=12)
        axes[idx].axis('off')

plt.tight_layout()

if 'google.colab' in sys.modules:
    save_path = '/content/drive/MyDrive/UAS_COMVIS/output/training_summary.png'
else:
    save_path = 'output/results/training_summary.png'
    os.makedirs('output/results', exist_ok=True)

plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved to: {save_path}")
plt.show()

## 7. Model Evaluation and Metrics

Load the best trained model and evaluate its performance on the validation dataset.

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

# Correct path pointing to the saved model inside your runs directory on Google Drive
best_model_path = Path('/content/drive/MyDrive/UAS_COMVIS/runs/detect/output/models/vehicle_detection/weights/best.pt')

if not best_model_path.exists():
    # Fallback to local working directory path structure if not in Google Drive
    best_model_path = Path('runs/detect/output/models/vehicle_detection/weights/best.pt')

print(f"Loading model from: {best_model_path}")
best_model = YOLO(str(best_model_path))
print("Best model loaded successfully!")

In [ ]:
metrics = best_model.val()

print("\nValidation Metrics:")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

names = data_config['names']
print("\nPer-class metrics:")
for i, class_name in enumerate(names):
    if i >= len(metrics.box.maps):
        continue
    map50_95 = float(metrics.box.maps[i])
    map50 = float(metrics.box.all_ap[i, 0]) if hasattr(metrics.box, 'all_ap') else float('nan')
    print(f"  {class_name}: mAP50={map50:.4f} | mAP50-95={map50_95:.4f}")


## 8. Vehicle Detection and Counting on Test Images

Apply the trained model on test images to detect and count vehicles. Results include bounding boxes, class labels, confidence scores, and total count per class.

In [ ]:
test_images_path = dataset_path / 'test' / 'images'
test_images = list(test_images_path.glob('*.*'))[:6] if test_images_path.exists() else []

if len(test_images) == 0:
    test_images = list((dataset_path / 'valid' / 'images').glob('*.*'))[:6]
    print("Using validation images for testing")

print(f"Found {len(test_images)} test images")

In [ ]:
detection_results = []

for img_path in test_images:
    results = best_model(img_path, conf=0.25)

    for result in results:
        img = result.orig_img.copy()
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        vehicle_counts = {class_name: 0 for class_name in data_config['names']}

        if result.boxes is not None and len(result.boxes) > 0:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                conf = box.conf[0].cpu().numpy()
                cls = int(box.cls[0].cpu().numpy())
                class_name = data_config['names'][cls]

                vehicle_counts[class_name] += 1

                cv2.rectangle(img_rgb, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
                label = f"{class_name}: {conf:.2f}"
                cv2.putText(img_rgb, label, (int(x1), int(y1)-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        detection_results.append({
            'image': img_path.name,
            'annotated_image': img_rgb,
            'counts': vehicle_counts,
            'total': sum(vehicle_counts.values())
        })

print(f"Detection completed on {len(detection_results)} images")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for idx, result in enumerate(detection_results[:6]):
    axes[idx].imshow(result['annotated_image'])
    axes[idx].axis('off')

    count_text = f"Total: {result['total']}\n"
    for class_name, count in result['counts'].items():
        if count > 0:
            count_text += f"{class_name}: {count}\n"

    axes[idx].set_title(count_text.strip(), fontsize=10)

plt.tight_layout()

if 'google.colab' in sys.modules:
    save_path = '/content/drive/MyDrive/UAS_COMVIS/output/detection_results.png'
else:
    save_path = 'output/results/detection_results.png'
    os.makedirs('output/results', exist_ok=True)

plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved to: {save_path}")
plt.show()

## 9. Detection Statistics and Summary

Generate summary statistics of detected vehicles across all test images.

In [ ]:
total_counts = {class_name: 0 for class_name in data_config['names']}

for result in detection_results:
    for class_name, count in result['counts'].items():
        total_counts[class_name] += count

print("Detection Summary:")
print(f"Total images processed: {len(detection_results)}")
print(f"\nVehicle counts:")
total_vehicles = 0
for class_name, count in total_counts.items():
    print(f"  {class_name}: {count}")
    total_vehicles += count
print(f"\nTotal vehicles detected: {total_vehicles}")

stats_df = pd.DataFrame([
    {'Image': r['image'], 'Motor': r['counts'].get('motor', 0),
     'Mobil': r['counts'].get('mobil', 0), 'Bus': r['counts'].get('bus', 0),
     'Truck': r['counts'].get('truck', 0), 'Total': r['total']}
    for r in detection_results
])

print("\nDetailed statistics:")
print(stats_df.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

classes = list(total_counts.keys())
counts = list(total_counts.values())
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

ax1.bar(classes, counts, color=colors, edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Vehicle Type')
ax1.set_ylabel('Count')
ax1.set_title('Total Vehicle Count by Type')
ax1.grid(axis='y', alpha=0.3)

for i, count in enumerate(counts):
    ax1.text(i, count + 0.5, str(count), ha='center', va='bottom', fontweight='bold')

if sum(counts) > 0:
    ax2.pie(counts, labels=classes, colors=colors, autopct='%1.1f%%',
            startangle=90, explode=[0.05]*len(classes))
    ax2.set_title('Vehicle Distribution')
else:
    ax2.text(0.5, 0.5, 'No vehicles detected', ha='center', va='center')

plt.tight_layout()
plt.savefig('output/results/vehicle_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save Detection Logs

Save detection results to JSON file for future reference and analysis.

In [ ]:
log_data = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model': 'YOLOv8n',
    'total_images': len(detection_results),
    'total_vehicles': total_vehicles,
    'vehicle_counts': total_counts,
    'metrics': {
        'mAP50': float(metrics.box.map50),
        'mAP50-95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr)
    },
    'detections': [
        {
            'image': r['image'],
            'counts': r['counts'],
            'total': r['total']
        }
        for r in detection_results
    ]
}

log_path = Path('output/logs/detection_log.json')
log_path.parent.mkdir(parents=True, exist_ok=True)

with open(log_path, 'w') as f:
    json.dump(log_data, f, indent=4)

print(f"Detection log saved to: {log_path}")
print(f"Total entries: {len(log_data['detections'])}")

## 11. Video/CCTV Detection (Optional)

Function to perform real-time detection on video files or CCTV streams. This can be used for traffic monitoring applications.

In [ ]:
def detect_video(video_path, output_path='output/detections/video_output.mp4', conf_threshold=0.25, max_frames=None):
    """
    Detect vehicles in video file or CCTV stream

    Parameters:
    - video_path: Path to video file or RTSP/HLS URL for CCTV stream
    - output_path: Path to save output video
    - conf_threshold: Confidence threshold for detection
    - max_frames: Maximum number of frames to process (None = all frames)
    """
    # For HLS/HTTP streams, may need special handling
    import os
    os.environ["OPENCV_FFMPEG_CAPTURE_OPTIONS"] = "rtsp_transport;udp"

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"ERROR: Cannot open video source: {video_path}")
        print("Make sure ffmpeg is installed and the URL is accessible")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    if fps == 0:
        fps = 25  # Default FPS for streams

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"Stream opened successfully")
    print(f"Resolution: {width}x{height} at {fps} FPS")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    detection_stats = {class_name: 0 for class_name in data_config['names']}

    print(f"Processing video: {video_path}")
    print("Processing frames...")

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                print("End of stream or cannot read frame")
                break

            # Check max frames limit
            if max_frames and frame_count >= max_frames:
                print(f"Reached max frames limit: {max_frames}")
                break

            results = best_model(frame, conf=conf_threshold, verbose=False)

            vehicle_counts = {class_name: 0 for class_name in data_config['names']}

            for result in results:
                if result.boxes is not None:
                    for box in result.boxes:
                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                        conf = box.conf[0].cpu().numpy()
                        cls = int(box.cls[0].cpu().numpy())
                        class_name = data_config['names'][cls]

                        vehicle_counts[class_name] += 1
                        detection_stats[class_name] += 1

                        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
                        label = f"{class_name}: {conf:.2f}"
                        cv2.putText(frame, label, (int(x1), int(y1)-10),
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            # Draw vehicle counts on frame
            y_offset = 30
            for class_name, count in vehicle_counts.items():
                text = f"{class_name}: {count}"
                cv2.putText(frame, text, (10, y_offset),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                y_offset += 30

            # Draw frame counter
            cv2.putText(frame, f"Frame: {frame_count}", (10, height - 20),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            out.write(frame)
            frame_count += 1

            if frame_count % 30 == 0:
                print(f"Processed {frame_count} frames - Total detected: {sum(detection_stats.values())}")

    except KeyboardInterrupt:
        print("\nProcessing interrupted by user")
    except Exception as e:
        print(f"\nError during processing: {str(e)}")
    finally:
        cap.release()
        out.release()

    print("\n" + "="*60)
    print("Video processing complete")
    print(f"Output saved to: {output_path}")
    print(f"Total frames processed: {frame_count}")
    print("\nTotal detections across all frames:")
    for class_name, count in detection_stats.items():
        print(f"  {class_name}: {count}")
    print("="*60)

print("Video detection function defined")
print("\nUsage examples:")
print("  Local file:  detect_video('path/to/video.mp4')")
print("  CCTV stream: detect_video('http://cctv-url/stream.m3u8', max_frames=300)")
print("  RTSP stream: detect_video('rtsp://camera-ip:554/stream')")

## 12. Model Export for Deployment

Export the trained model in different formats for deployment in production environments.

In [ ]:
export_formats = ['onnx', 'torchscript']

print("Exporting model to different formats...")

for fmt in export_formats:
    try:
        export_path = best_model.export(format=fmt)
        print(f"{fmt.upper()} export successful: {export_path}")
    except Exception as e:
        print(f"{fmt.upper()} export failed: {str(e)}")

# Correct path inside the runs directory where best.pt is stored
source_best_pt = 'runs/detect/output/models/vehicle_detection/weights/best.pt'
best_model_path = 'output/models/yolov8_trained.pt'

os.makedirs(os.path.dirname(best_model_path), exist_ok=True)
shutil.copy(source_best_pt, best_model_path)
print(f"\nBest model successfully copied to: {best_model_path}")

## 13. Project Summary

Summary of the vehicle detection project including model performance and key metrics.

In [ ]:
print("=" * 60)
print("VEHICLE DETECTION PROJECT SUMMARY")
print("=" * 60)
print(f"\nAlgorithm: YOLOv8 (You Only Look Once)")
print(f"CNN Architecture: YOLOv8 Nano")
print(f"Dataset: Custom Vehicle Dataset from Roboflow")
print(f"Classes: {', '.join(data_config['names'])}")
print(f"\nModel Performance:")
print(f"  - mAP50: {metrics.box.map50:.4f}")
print(f"  - mAP50-95: {metrics.box.map:.4f}")
print(f"  - Precision: {metrics.box.mp:.4f}")
print(f"  - Recall: {metrics.box.mr:.4f}")
print(f"\nDetection Results:")
print(f"  - Total images processed: {len(detection_results)}")
print(f"  - Total vehicles detected: {total_vehicles}")
print(f"\nVehicle Distribution:")
for class_name, count in total_counts.items():
    percentage = (count / total_vehicles * 100) if total_vehicles > 0 else 0
    print(f"  - {class_name}: {count} ({percentage:.1f}%)")
print(f"\nModel Files:")
print(f"  - Best model: output/models/yolov8_trained.pt")
print(f"  - Training results: output/models/vehicle_detection/")
print(f"  - Detection logs: output/logs/detection_log.json")
print(f"\nApplications:")
print(f"  - Traffic monitoring and counting")
print(f"  - CCTV surveillance")
print(f"  - Parking management")
print(f"  - Vehicle type classification")
print("=" * 60)

## 14. Real-Time CCTV Detection Test

Test the model on live CCTV stream from Jogja city traffic camera.

In [ ]:
# CCTV Stream URL from Jogja City
cctv_url = "https://cctvjss.jogjakota.go.id/kotabaru/ANPR-Jl-Ahmad-Jazuli.stream/playlist.m3u8"

# Output path
output_video = 'output/detections/cctv_jogja_detection.mp4'
os.makedirs('output/detections', exist_ok=True)

print("="*60)
print("OPTION 1: SAVE TO FILE (NO DISPLAY)")
print("="*60)
print(f"Stream URL: {cctv_url}")
print("\nNOTE: Make sure you have ffmpeg installed:")
print("  Mac: brew install ffmpeg")
print("  Or download from: https://ffmpeg.org/download.html")
print("\nThis will SAVE video to file without live preview")
print("For LIVE PREVIEW, use the next cell (Cell 43)")
print("Press Ctrl+C in terminal to stop early if needed")
print("-" * 60)

# Run detection - SAVES TO FILE ONLY
# Uncomment line below to start (process 300 frames = ~10 seconds)
# detect_video(cctv_url, output_video, conf_threshold=0.3, max_frames=300)

# Or process more frames:
detect_video(cctv_url, output_video, conf_threshold=0.3, max_frames=1500)

## 15. Live CCTV Streaming with Real-Time Display

Real-time detection with live preview window (for local Mac only, not for Colab).

In [ ]:
def detect_video_live(video_path,
                      conf_threshold=0.2,
                      iou_threshold=0.5,
                      imgsz=1280,
                      save_output=False,
                      output_path='output/detections/cctv_live.mp4',
                      max_frames=None,
                      count_line_ratio=0.60):
    """
    Real-time vehicle detection + tracking on video / CCTV stream.

    Perbaikan dibanding versi sebelumnya:
    - ByteTrack (bukan centroid distance 50 px) -> ID stabil
    - imgsz default 1280 -> kendaraan kecil tetap kebaca
    - conf default 0.2 -> motor malam tidak kebuang
    - line-crossing counter -> hitung kendaraan lewat, bukan jumlah box/frame
    - aman di Colab (tidak memanggil cv2.imshow)
    """
    import os
    IN_COLAB = 'google.colab' in sys.modules
    os.environ["OPENCV_FFMPEG_CAPTURE_OPTIONS"] = "rtsp_transport;udp"

    if 'best_model' not in globals():
        print("ERROR: best_model belum ada. Jalankan cell training / load model dulu.")
        return

    class_names = data_config['names'] if 'data_config' in globals() else best_model.names
    if isinstance(class_names, dict):
        class_names = [class_names[i] for i in sorted(class_names)]

    if IN_COLAB and not save_output:
        save_output = True
        print("Colab terdeteksi: output video akan disimpan (cv2.imshow tidak tersedia).")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"ERROR: tidak bisa membuka source: {video_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    fps = int(fps) if fps and fps > 1 else 25
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Stream opened: {width}x{height} @ {fps} FPS | classes={class_names}")
    print(f"imgsz={imgsz} conf={conf_threshold} iou={iou_threshold} tracker=bytetrack")

    out = None
    if save_output:
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'),
                              fps, (width, height))

    frame_count = 0
    seen_ids = {name: set() for name in class_names}
    crossed_ids = {name: set() for name in class_names}
    last_cy = {}
    line_y = int(height * count_line_ratio)

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                print("End of stream / frame tidak terbaca")
                break

            if max_frames and frame_count >= max_frames:
                print(f"Reached max_frames={max_frames}")
                break

            results = best_model.track(
                frame,
                persist=True,
                tracker='bytetrack.yaml',
                imgsz=imgsz,
                conf=conf_threshold,
                iou=iou_threshold,
                max_det=300,
                verbose=False,
            )
            r = results[0]
            current_counts = {name: 0 for name in class_names}

            if r.boxes is not None and len(r.boxes) > 0:
                xyxy = r.boxes.xyxy.cpu().numpy()
                confs = r.boxes.conf.cpu().numpy()
                clss = r.boxes.cls.cpu().numpy().astype(int)
                ids = r.boxes.id.int().cpu().tolist() if r.boxes.id is not None else [None] * len(xyxy)

                for (x1, y1, x2, y2), conf, cls, tid in zip(xyxy, confs, clss, ids):
                    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
                    current_counts[name] = current_counts.get(name, 0) + 1

                    if tid is not None:
                        seen_ids.setdefault(name, set()).add(tid)
                        cy = (y1 + y2) / 2
                        prev = last_cy.get(tid)
                        if prev is not None and (prev < line_y <= cy or prev > line_y >= cy):
                            crossed_ids.setdefault(name, set()).add(tid)
                        last_cy[tid] = cy

                    color = (0, 255, 0)
                    cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
                    label = f"{name} {conf:.2f}" + (f" #{tid}" if tid is not None else "")
                    cv2.putText(frame, label, (int(x1), max(15, int(y1) - 8)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            cv2.line(frame, (0, line_y), (width, line_y), (0, 255, 255), 2)

            panel = frame.copy()
            cv2.rectangle(panel, (5, 5), (430, 40 + len(class_names) * 26), (0, 0, 0), -1)
            frame = cv2.addWeighted(frame, 0.75, panel, 0.25, 0)

            cv2.putText(frame, f"Frame {frame_count}", (12, 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
            y = 56
            for name in class_names:
                text = (f"{name}: now {current_counts.get(name, 0)} | "
                        f"lewat {len(crossed_ids.get(name, set()))} | "
                        f"unik {len(seen_ids.get(name, set()))}")
                cv2.putText(frame, text, (12, y), cv2.FONT_HERSHEY_SIMPLEX,
                            0.55, (255, 255, 255), 2)
                y += 26

            if out is not None:
                out.write(frame)

            if IN_COLAB and frame_count % 25 == 0:
                try:
                    from google.colab.patches import cv2_imshow
                    cv2_imshow(frame)
                except Exception:
                    pass

            if not IN_COLAB:
                cv2.imshow('CCTV Vehicle Detection - Q to quit', frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    print("Stopped by user (Q)")
                    break

            frame_count += 1
            if frame_count % 50 == 0:
                total_now = sum(current_counts.values())
                print(f"Frame {frame_count} | visible={total_now} | " +
                      " | ".join(f"{n}:{len(seen_ids.get(n, set()))}" for n in class_names))

    except KeyboardInterrupt:
        print("Interrupted by user")
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
    finally:
        cap.release()
        if out is not None:
            out.release()
        if not IN_COLAB:
            cv2.destroyAllWindows()

    print("=" * 60)
    print(f"Frames processed: {frame_count}")
    print("Unique IDs seen (per class):")
    for name in class_names:
        print(f"  {name}: seen={len(seen_ids.get(name, set()))} "
              f"crossed_line={len(crossed_ids.get(name, set()))}")
    if save_output:
        print(f"Output saved: {output_path}")
    print("=" * 60)

print("Live detection (ByteTrack) ready")


In [ ]:
# STEP 1: Load trained model
print("Loading trained model...")
try:
    best_model = YOLO('/content/drive/MyDrive/UAS_COMVIS/runs/detect/output/models/vehicle_detection/weights/best.pt')
    print("Model loaded successfully!")
    print(f"Model classes: {best_model.names}")
except Exception as e:
    print(f"ERROR: Cannot load model - {e}")
    print("Jalankan cell training dulu, atau load model dari Google Drive.")
    raise

# STEP 2: Live detection
cctv_url = "https://cctvjss.jogjakota.go.id/kotabaru/ANPR-Jl-Ahmad-Jazuli.stream/playlist.m3u8"

print("\n" + "=" * 60)
print("LIVE CCTV DETECTION - JOGJA (MALIOBORO)")
print("=" * 60)
print(f"Stream: {cctv_url}")
print("Setting: imgsz=1280, conf=0.2, iou=0.5, tracker=ByteTrack")
print("Colab tidak bisa membuka window -> hasil disimpan ke output/detections/")
print("Tekan tombol Stop / Ctrl+C untuk berhenti.")
print("=" * 60)

import time
time.sleep(2)

detect_video_live(
    cctv_url,
    conf_threshold=0.2,
    iou_threshold=0.5,
    imgsz=1280,
    save_output=True,
    output_path='output/detections/cctv_jogja_live.mp4',
    max_frames=900,
)

# Setelah selesai, download videonya:
from google.colab import files
files.download('output/detections/cctv_jogja_live.mp4')
